# Phase 1b — GPT-2 SFT baselines on Kaggle (GPU)

Trains the **CoT-SFT** reproduction gate (and optionally No-CoT-SFT) on GPT-2 over
GSM8k-Aug, then evaluates exact-match on GSM8k + OOD sets.

**Settings (right panel):**
- Accelerator: **GPU T4 x2** or **P100**.
- Internet: **On** (first run downloads GPT-2 + datasets from HuggingFace).

**Surviving the session cap:** the trainer checkpoints to `outputs/…/checkpoints/`. If a
session ends before `total_steps`, it exits with code 42; re-running any training cell
resumes from the latest checkpoint (see the resume section).

## 1. Code + dependencies

In [ ]:
import os, subprocess

# Configure Hub transfers before any Hugging Face library is imported.
# Add a Kaggle secret named HF_TOKEN (a read token is sufficient) to avoid
# anonymous rate limits and public CDN signing failures.
os.environ["HF_HUB_DISABLE_XET"] = "1"
os.environ["HF_HUB_DOWNLOAD_TIMEOUT"] = "300"
try:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    print("Hugging Face authentication: Kaggle secret loaded")
except Exception:
    print("Hugging Face authentication: no HF_TOKEN secret; using public access")

REPO_URL = "https://github.com/0x0shephard/latent-reasoning.git"
REPO_DIR = "/kaggle/working/latent-reasoning"

if os.path.isdir(os.path.join(REPO_DIR, ".git")):
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", "--branch", "main", REPO_URL, REPO_DIR], check=True)
os.chdir(REPO_DIR)

!pip install -q -r requirements.txt
import torch; print("torch", torch.__version__, "| CUDA:", torch.cuda.is_available())
print("commit:", subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip())

## 2. (Optional) offline data

Skip this on the first run (Internet=On downloads directly). To run fully offline:
stage data locally with `python scripts/dataset_prep.py`, upload the `hf_cache` folder as
a Kaggle Dataset, attach it, and uncomment the environment settings below.

In [ ]:
# os.environ["HF_HOME"] = "/kaggle/input/<your-dataset>/hf_cache"
# os.environ["CODIKAVA_DATA_ROOT"] = "/kaggle/input/<your-dataset>/hf_cache/prepared"
# os.environ["HF_HUB_OFFLINE"] = "1"
# os.environ["TRANSFORMERS_OFFLINE"] = "1"

## 3. Validate the real Phase-1 contract

This resolves the actual tokenizer and datasets, checks their schemas, measures CoT
truncation, verifies answer parsing, and rejects exact train/eval question leakage. Do
not spend a GPU training session unless this cell reports `status: ok`.

In [ ]:
!python scripts/validate_phase1.py --config configs/sft_cot.yaml

## 4. Train CoT-SFT (the reproduction gate)

Run this cell as many times as needed — each run resumes from the latest checkpoint and
stops at the wall-clock guard or `total_steps`. To finish faster for a first look, lower
steps with `--set train.total_steps=3000`.

In [ ]:
!python -m src.train.kaggle_run --config configs/sft_cot.yaml

## 5. Evaluate

Greedy decode + exact-match on GSM8k (in-domain) and SVAMP / MultiArith / GSM-Hard (OOD).
Use `--limit` for a quick check before a full sweep. Predictions and a JSON summary
are written beneath `outputs/sft_cot/eval/step_XXXXXXXX/`.

In [ ]:
!python -m src.eval.run_eval --config configs/sft_cot.yaml --limit 200

## 6. (Optional) No-CoT-SFT baseline

The lower bound — direct answer, no reasoning. Same commands, different config.

In [ ]:
# !python -m src.train.kaggle_run --config configs/sft_nocot.yaml
# !python -m src.eval.run_eval --config configs/sft_nocot.yaml --limit 200

## Resuming past the session cap

Checkpoints live under `outputs/…/checkpoints/` in `/kaggle/working`. To continue in a
**new** session:
1. **Save Version (Commit)** so `/kaggle/working` persists as this notebook's output.
2. In the new session, attach that output as an input and copy the checkpoints back into
   `outputs/…/checkpoints/` (or keep working in the same persisted notebook).
3. Re-run the training cell — it prints `[resume] continuing from step N` and picks up
   exactly where it stopped (guaranteed by the Phase 0 resume tests).

**Phase 1 exit gate:** CoT-SFT reaches sane GSM8k exact-match and eval reproduces the same
number across two runs on a fixed checkpoint. Then we move to Phase 2 (CODI + KaVa).